# The `dagger` modifier

**Download this notebook - {nb-download}`dagger.ipynb`**

A `with dagger:` block applies the inverse of its quantum operations: it reverses
their order and replaces each gate with its inverse.

Here we prepare $(|0\rangle + i|1\rangle)/\sqrt{2}$, undo that preparation,
and check that the emulator returns the qubit to $|0\rangle$.

In [1]:
import numpy as np

from guppylang import guppy
from guppylang.std.builtins import dagger, output
from guppylang.std.debug import state_output
from guppylang.std.quantum import h, measure, qubit, s

In [2]:
@guppy(daggerable=True)
def prepare(q: qubit) -> None:
    h(q)
    s(q)


@guppy
def unprepare() -> None:
    q = qubit()
    prepare(q)
    state_output("prepared", q)

    with dagger:
        prepare(q)

    state_output("restored", q)
    output("result", measure(q).read())

`daggerable=True` allows `prepare` to be called inside a `dagger` block.
The forward call applies `h` followed by `s`. The daggered call applies the inverse
of `s` first, then `h`, which is its own inverse.

The statevector simulator lets us verify both the prepared state and the restored
state, including their relative phases. A global phase does not change the state.
The final measurement must be zero.

In [3]:
shots = unprepare.emulator(n_qubits=1).statevector_sim().run()
states = shots.partial_state_dicts()[0]
prepared = states["prepared"].as_single_state()
restored = states["restored"].as_single_state()

expected = np.array([1, 1j]) / np.sqrt(2)
assert np.isclose(abs(np.vdot(expected, prepared)), 1)
assert np.isclose(abs(np.vdot([1, 0], restored)), 1)
result = shots[0].as_dict()["result"]
assert result == 0
print("Measurement after unpreparing:", int(result))

Measurement after unpreparing: 0
